In [1]:
import pandas as pd
import os
from sqlalchemy import create_engine, text

In [2]:
DB_HOST = os.getenv("DB_HOST", "db")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "weather_db")
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "postgres")

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

In [3]:
DATABASE_URL

'postgresql+psycopg2://postgres:postgres@db:5432/weather_db'

In [4]:
engine = create_engine(DATABASE_URL)

In [5]:
with engine.connect() as conn:
    loc_df = pd.read_sql("select * from location", conn)
    weather_df = pd.read_sql("select * from weather_observation", conn)

In [6]:
loc_df

,id,city,state,country,latitude,longitude,created_at
0,bc840c34-2c64-4396-bca3-6207a9a0096b,Seattle,WA,USA,47.6062,-122.3321,2026-05-15 07:17:30.499367+00:00
1,8661e90e-c38e-4828-bada-8c2493c4f94b,Bothell,WA,USA,47.7623,-122.2054,2026-05-15 07:17:30.499367+00:00
2,252e7575-ecc0-479d-ba60-0e90e5b5c30e,Portland,OR,USA,45.5152,-122.6784,2026-05-15 07:17:30.499367+00:00


In [7]:
weather_df.head()

,id,location_id,observed_at,temperature,feels_like,humidity,pressure,wind_speed,weather_description,weather_code,raw_json,created_at
0,7f0fa015-cbea-47d2-aaab-ae719e230b80,bc840c34-2c64-4396-bca3-6207a9a0096b,2026-05-06 00:00:00+00:00,17.7,17.3,70.0,None,7.6,None,3,None,2026-05-15 07:17:31.227774+00:00
1,e6113d1b-776d-4e65-8e38-ce61e9aaab80,bc840c34-2c64-4396-bca3-6207a9a0096b,2026-05-06 01:00:00+00:00,17.3,16.9,72.0,None,7.8,None,1,None,2026-05-15 07:17:31.227774+00:00
2,9f465932-9ea3-4ca1-868c-c4761d7e5eaf,bc840c34-2c64-4396-bca3-6207a9a0096b,2026-05-06 02:00:00+00:00,17.9,17.4,67.0,None,7.2,None,0,None,2026-05-15 07:17:31.227774+00:00
3,49e89da8-608b-4a83-81f1-6089760b42c7,bc840c34-2c64-4396-bca3-6207a9a0096b,2026-05-06 03:00:00+00:00,16.3,15.8,72.0,None,6.2,None,0,None,2026-05-15 07:17:31.227774+00:00
4,f86b3856-d591-40a3-a02a-170d6ed45f14,bc840c34-2c64-4396-bca3-6207a9a0096b,2026-05-06 04:00:00+00:00,15.0,14.3,77.0,None,6.8,None,0,None,2026-05-15 07:17:31.227774+00:00


In [8]:
weather_df.dtypes

id                                  object
location_id                         object
observed_at            datetime64[us, UTC]
temperature                        float64
feels_like                         float64
humidity                           float64
pressure                            object
wind_speed                         float64
weather_description                 object
weather_code                         int64
raw_json                            object
created_at             datetime64[us, UTC]
dtype: object

In [9]:
combined = loc_df.merge(weather_df, left_on='id', right_on='location_id', suffixes=['_loc', '_wea'])

In [10]:
combined.head()

,id_loc,city,state,country,latitude,longitude,created_at_loc,id_wea,location_id,observed_at,temperature,feels_like,humidity,pressure,wind_speed,weather_description,weather_code,raw_json,created_at_wea
0,bc840c34-2c64-4396-bca3-6207a9a0096b,Seattle,WA,USA,47.6062,-122.3321,2026-05-15 07:17:30.499367+00:00,7f0fa015-cbea-47d2-aaab-ae719e230b80,bc840c34-2c64-4396-bca3-6207a9a0096b,2026-05-06 00:00:00+00:00,17.7,17.3,70.0,None,7.6,None,3,None,2026-05-15 07:17:31.227774+00:00
1,bc840c34-2c64-4396-bca3-6207a9a0096b,Seattle,WA,USA,47.6062,-122.3321,2026-05-15 07:17:30.499367+00:00,e6113d1b-776d-4e65-8e38-ce61e9aaab80,bc840c34-2c64-4396-bca3-6207a9a0096b,2026-05-06 01:00:00+00:00,17.3,16.9,72.0,None,7.8,None,1,None,2026-05-15 07:17:31.227774+00:00
2,bc840c34-2c64-4396-bca3-6207a9a0096b,Seattle,WA,USA,47.6062,-122.3321,2026-05-15 07:17:30.499367+00:00,9f465932-9ea3-4ca1-868c-c4761d7e5eaf,bc840c34-2c64-4396-bca3-6207a9a0096b,2026-05-06 02:00:00+00:00,17.9,17.4,67.0,None,7.2,None,0,None,2026-05-15 07:17:31.227774+00:00
3,bc840c34-2c64-4396-bca3-6207a9a0096b,Seattle,WA,USA,47.6062,-122.3321,2026-05-15 07:17:30.499367+00:00,49e89da8-608b-4a83-81f1-6089760b42c7,bc840c34-2c64-4396-bca3-6207a9a0096b,2026-05-06 03:00:00+00:00,16.3,15.8,72.0,None,6.2,None,0,None,2026-05-15 07:17:31.227774+00:00
4,bc840c34-2c64-4396-bca3-6207a9a0096b,Seattle,WA,USA,47.6062,-122.3321,2026-05-15 07:17:30.499367+00:00,f86b3856-d591-40a3-a02a-170d6ed45f14,bc840c34-2c64-4396-bca3-6207a9a0096b,2026-05-06 04:00:00+00:00,15.0,14.3,77.0,None,6.8,None,0,None,2026-05-15 07:17:31.227774+00:00


In [11]:
combined.groupby('city')['temperature'].mean()

city
Bothell     13.735833
Portland    15.210833
Seattle     13.050833
Name: temperature, dtype: float64

In [12]:
combined.groupby('city')['feels_like'].mean()

city
Bothell     12.984167
Portland    14.781667
Seattle     11.976667
Name: feels_like, dtype: float64